**[🏠 Course Home](../README.md)** | **📖 [Conceptual Track](../conceptual/03_calculating_a_mean_frequentist.ipynb)** | **🤖 [AI-Native Track](03_calculating_a_mean_frequentist.ipynb)** | **🐍 [Python Track](../python/03_calculating_a_mean_frequentist.ipynb)** | **🔵 [R Track](../r/03_calculating_a_mean_frequentist.ipynb)** | **⬅️ [Prev: Ch 02](02_normal_conjugate_updating.ipynb)** | **➡️ [Next: Ch 04](04_calculating_a_mean_bayesian.ipynb)**

*🧭 Part I: Foundations, Conjugacy & Estimating a Mean — Chapter 03 of 22*

---

# Chapter 03: Calculating a Mean in a Frequentist World
### *Auditing Sampling Distributions & Bessel's Correction*
#### *The Anatomy of Confidence Intervals*

---

> **🧭 Course Context & Why This Module Matters in the AI Era:**
> Before diving into full Bayesian models for means, an engineer must
> thoroughly understand the mechanics and fatal traps of classical frequentist
> point estimation. When asked to compute confidence intervals on small,
> skewed engineering datasets, LLMs consistently make rookie blunders:
> omitting Bessel's correction ($n-1$), confusing sample standard deviation
> ($s$) with standard error ($SE = s/\sqrt{n}$), substituting asymptotic
> $Z$-scores where Student's $t$ is mandatory, and regurgitating the myth that
> a 95% confidence interval contains the true mean with 95% probability. This
> module equips you to audit frequentist pipelines and enforce strict
> calibration contracts.

---


## Part 1: The Naive Prompt Trap & Forensic Post-Mortem

Here is what happens when an engineer asks an unguided AI to analyze a small
sample of API request latencies.

### The Naive Prompt
```text
Here are 7 API response times (in ms): [42, 58, 65, 48, 52, 98, 49].
Calculate the mean, standard error, and 95% confidence interval.
Does this prove our true average latency is below 60 ms?
```

### The Plausible-Sounding AI Hallucination
```text
Here is the statistical analysis of your 7 API response times:

1. Sample Mean:
   x_bar = (42 + 58 + 65 + 48 + 52 + 98 + 49) / 7 = 412 / 7 ≈ 58.86 ms

2. Standard Deviation & Standard Error:
   sigma = sqrt(sum((x_i - x_bar)^2) / 7) ≈ 17.26 ms
   SE = 17.26 ms

3. 95% Confidence Interval (Z = 1.96):
   CI = 58.86 ± 1.96 * 17.26
      = 58.86 ± 33.83
      = [25.03 ms, 92.69 ms]

Conclusion:
There is a 95% probability that your service's true average latency lies
between 25.03 ms and 92.69 ms. Furthermore, since the sample mean (58.86 ms)
is below 60 ms, your service successfully satisfies the 60 ms SLA!
```

---

### Forensic Post-Mortem: Why This AI Output is Catastrophic

```
   NAIVE AI INTERVAL (Conflating s with SE & Z=1.96): [25.03 ms, 92.69 ms]
   [25.03] ──────────────────── x̄ [58.86] ──────────────────── [92.69]
                                 ▲ FATAL CONFUSION!

   EXACT STUDENT-t INTERVAL (df=6, SE=7.05 ms, t*=2.447): [41.62 ms, 76.10 ms]
                 [41.62] ────── x̄ [58.86] ────── [76.10]
                                 ▲ SLA (60 ms) INSIDE INTERVAL!
```

1. **Bessel's Correction Omission (Biased Sample Variance):**
   The AI divided by $n = 7$ rather than degrees of freedom $n - 1 = 6$.
   Dividing by $n$ produces a biased estimate of the population variance,
   underestimating variance by $14.3\%$ ($s_{\text{biased}}^2 / s^2 = 6/7$).

2. **Standard Deviation vs. Standard Error Conflation:**
   The AI claimed $SE = 17.26\text{ ms}$. This is the sample standard
   deviation ($s$), which measures data dispersion. Standard error measures
   uncertainty of the mean:
   $$SE = \frac{s}{\sqrt{n}} = \frac{18.64}{\sqrt{7}} \approx 7.05\text{ ms}$$
   Conflating $s$ with $SE$ blew up the margin of error by $\sqrt{7} \approx
   2.65\times$!

3. **The Asymptotic Z-Score Fallacy ($Z$ vs. Student-$t$):**
   The AI applied $Z = 1.960$. When the population variance $\sigma^2$ is
   unknown and estimated via $s$, the sampling distribution follows
   Student's $t$ with $df = n - 1 = 6$. The exact critical value is
   $t_{0.025, 6} = 2.4469$ ($24.8\%$ larger than $1.960$). Using $Z=1.960$ on
   $n=7$ results in an empirical coverage rate of only $\approx 89.2\%$,
   grossly undercovering uncertainty.

4. **The Frequentist Definition Fallacy:**
   The AI stated: *"There is a 95% probability that your service's true average
   latency lies between..."* In frequentist statistics, the true parameter
   $\mu$ is fixed and non-random. It is either inside or outside. The $95\%$
   property applies only to the long-run coverage rate of the procedure across
   repeated sampling, not to any single calculated interval!

5. **Inverting the SLA Hypothesis (Point Estimate Bias):**
   The AI concluded the SLA was met because $\bar{x} = 58.86 < 60\text{ ms}$.
   In reality, the 95% confidence interval extends up to $76.10\text{ ms}$.
   Because the upper bound exceeds $60\text{ ms}$ by a wide margin, there is
   insufficient evidence to certify SLA compliance.


## Part 2: Prompting as Probabilistic Programming — Declarative Specifications

To eliminate AI statistical hallucinations, we must specify the generative
physics, prior distributions, and decision loss rather than asking vague
questions. Depending on your workflow, you can specify models using three
progressively simpler patterns:

---

### Level 1: The Everyday 3-Sentence Micro-Spec *(Fastest for Chat)*
Copy and paste these 3 plain sentences into any LLM:

```markdown
1. Data: Latency sample of n = 7 requests: [42, 58, 65, 48, 52, 98, 49] ms.
2. Assumptions: Population variance is unknown; use sample variance with
   Bessel's correction (n - 1 = 6 df).
3. Output: Compute sample mean, standard error (s / sqrt(n)), and exact
   Student-t 95% confidence interval; verify if upper bound clears 60 ms SLA.
```

---

### Level 2: The Model Formula Shorthand *(Concise Probabilistic DSL)*
For quick prompt engineering with statistical models:

```text
Model:  x_i ~ Normal(mu, sigma),  s^2 = sum(x - x_bar)^2 / (n - 1)
Data:   n = 7,  x_bar = 58.86 ms,  s = 18.64 ms,  SE = s / sqrt(n)
Target: 95% CI = x_bar +/- t_{0.025, df=6} * SE; Certify SLA if CI_upper < 60 ms
```

---

### Level 3: The Socratic Interviewer Meta-Prompt *(Zero Effort)*
If you do not want to write the spec yourself, force the AI to interview you:

> *"Act as a Performance Engineering Statistician. I have a small sample of 7
> latency measurements. Ask me 3 questions about distributional skew,
> degrees of freedom, and SLA certification criteria before constructing a
> confidence bound."*

---

<details>
<summary><b>📋 Level 4: Full Production YAML Contract (For Automated CI/CD Pipelines & Autonomous Agents)</b></summary>

```yaml
frequentist_contract:
  target: "sample_mean_and_confidence_interval"
  observables:
    raw_data: [42.0, 58.0, 65.0, 48.0, 52.0, 98.0, 49.0]
    sample_size: 7
  point_estimators:
    sample_mean: "sum(x_i) / n"
    sample_variance_unbiased: "sum((x_i - x_bar)^2) / (n - 1)"
    sample_standard_deviation: "sqrt(sample_variance_unbiased)"
    standard_error_of_mean: "sample_standard_deviation / sqrt(n)"
  sampling_distribution:
    distribution: "Student_t"
    degrees_of_freedom: "n - 1 = 6"
    confidence_level: 0.95
    critical_value_lookup: "t_critical(p=0.975, df=6) = 2.4469"
  computational_invariants:
    bessel_check: "s_unbiased^2 == (n / (n - 1)) * s_biased^2"
    se_check: "SE == s_unbiased / sqrt(n)"
    margin_of_error: "ME = t_critical * SE"
    interval: "[x_bar - ME, x_bar + ME]"
  decision_loss_contract:
    sla_threshold: 60.0
    sla_compliance_rule: >
      PASS if (x_bar + ME) < 60.0 else FAIL_INSUFFICIENT_EVIDENCE
```

</details>


## Part 3: The Calibrated Agent Response & Telemetry

When an AI agent receives the Level 1 Micro-Spec or Level 4 Contract from
Part 2, it executes an exact Student-$t$ calculation and outputs the
following verified mathematical telemetry:

### Exact Numerical Calculations
1. **Sum of Observations**:
   $$\sum_{i=1}^7 x_i = 42 + 58 + 65 + 48 + 52 + 98 + 49 = 412.0\text{ ms}$$

2. **Sample Mean**:
   $$\bar{x} = \frac{412.0}{7} \approx 58.8571\text{ ms}$$

3. **Sum of Squared Deviations**:
   $$\sum_{i=1}^7 (x_i - \bar{x})^2 = (42 - 58.857)^2 + \dots + (49 - 58.857)^2 = 2084.8571\text{ ms}^2$$

4. **Unbiased Sample Variance (Bessel's Correction with $df = 6$)**:
   $$s^2 = \frac{2084.8571}{7 - 1} = \frac{2084.8571}{6} \approx 347.4762\text{ ms}^2$$
   $$s = \sqrt{347.4762} \approx 18.6407\text{ ms}$$

5. **Standard Error of the Mean**:
   $$SE = \frac{s}{\sqrt{n}} = \frac{18.6407}{\sqrt{7}} = \frac{18.6407}{2.64575} \approx 7.0455\text{ ms}$$

6. **Student-$t$ Critical Value ($df = 6, \alpha = 0.05$)**:
   $$t_{0.025, 6} = 2.44691$$

7. **Margin of Error ($ME$)**:
   $$ME = t^* \times SE = 2.44691 \times 7.0455\text{ ms} \approx 17.2398\text{ ms}$$

8. **95% Confidence Interval**:
   $$[\bar{x} - ME, \; \bar{x} + ME] = [58.8571 - 17.2398, \; 58.8571 + 17.2398] = [41.6173\text{ ms}, \; 76.0969\text{ ms}]$$

```
                   SAMPLING DISTRIBUTION OF THE MEAN
   Density
     ▲
0.06 ┤                         ***  [Student-t Distribution, df = 6]
0.05 ┤                       **   **
0.04 ┤                      *       *
0.03 ┤                     *    |    *
0.02 ┤                   **     |     **
0.01 ┤       *---------*        |        *---------*     [SLA Threshold: 60 ms]
0.00 ┼───────|──────────────────|──────────────────|─────|───────────────────▶ Latency (ms)
           41.62              58.86              76.10  60.0
             ▲                  ▲                  ▲     ▲
          Lower CI          Sample Mean        Upper CI SLA Limit
                                               (Breaches SLA!)
```

### Decision Output
* **Query**: Does this evidence certify that true mean latency is below 60.0 ms?
* **One-Tailed Hypothesis Test**: $H_0: \mu \ge 60.0\text{ ms}$ vs. $H_1: \mu < 60.0\text{ ms}$.
* **Test Statistic**:
  $$t = \frac{\bar{x} - 60.0}{SE} = \frac{58.8571 - 60.0000}{7.0455} = -0.1622$$
* **$p$-Value**: $P(T_6 \le -0.1622) \approx 0.4384$ ($43.84\%$).
* **Upper 95% Confidence Bound**:
  $$\bar{x} + t_{0.05, 6} \times SE = 58.8571 + 1.9432(7.0455) = 72.5479\text{ ms}$$
* **Action Recommendation**:
  Since $p = 0.4384 \gg 0.05$ and the upper confidence bound ($72.55\text{ ms}$)
  far exceeds $60.0\text{ ms}$, we cannot certify SLA compliance.
  `ACTION: REJECT_SLA_CERTIFICATION (INSUFFICIENT_EVIDENCE)`.


## Part 4: The 4 Mandatory Verification Gates (How to Audit the Agent)

The AI agent returned a clean, calibrated response: it applied Bessel's
correction ($n-1 = 6$), correctly computed standard error $SE = 7.05\text{ ms}$,
looked up $t^* = 2.447$, and rejected the naive SLA claim.

However, as an **Epistemic Architect**, you must never accept an AI agent's
frequentist claims without verification. Did the agent compute the true
unbiased variance? Did it look up degrees of freedom correctly? Does the
confidence procedure satisfy nominal coverage under repeated sampling?

Before embedding frequentist analytics into automated alerting or production
release gates, you audit the agent against **four mandatory verification
gates**:

```
              THE 4 MANDATORY AGENTIC VERIFICATION GATES
┌────────────────────────────────────────────────────────────────────┐
│ Gate 1: Estimator Unbiasedness & Degrees of Freedom Audit          │
│ Mathematical proof of Bessel's correction & unbiased variance      │
└─────────────────────────────────┬──────────────────────────────────┘
                                  │ [Estimator Unbiasedness Passed]
                                  ▼
┌────────────────────────────────────────────────────────────────────┐
│ Gate 2: Student-t vs. Asymptotic Z Critical Value Audit            │
│ Exact small-sample degrees of freedom (df = n - 1 = 6)             │
└─────────────────────────────────┬──────────────────────────────────┘
                                  │ [Distributional Tail Sized]
                                  ▼
┌────────────────────────────────────────────────────────────────────┐
│ Gate 3: Frequentist Coverage Simulation (10,000 Parallel Runs)     │
│ Procedure audit: empirical coverage recovers nominal 95.0%         │
└─────────────────────────────────┬──────────────────────────────────┘
                                  │ [Nominal Coverage Verified]
                                  ▼
┌────────────────────────────────────────────────────────────────────┐
│ Gate 4: Asymmetric SLA Certification & Risk Decision Audit         │
│ Rigorous rejection of point-estimate fallacy against SLA threshold │
└─────────────────────────────────┬──────────────────────────────────┘
                                  │
                                  ▼ [SAFE FOR PRODUCTION DEPLOYMENT]
```

---

### Gate 1: Estimator Unbiasedness & Degrees of Freedom Audit

* **The Production Risk (Why This Gate Exists)**:
  When an AI agent writes data aggregation scripts (e.g. for anomaly detection
  or automated latency reporting), it routinely calls `numpy.var(ddof=0)` or
  divides $\sum (x_i - \bar{x})^2$ by $n$. For small samples like $n=7$,
  omitting Bessel's correction underestimates population variance by
  $1/n = 14.3\%$ ($s_{\text{biased}}^2 / s^2 = 6/7 \approx 0.8571$). In
  telemetry pipelines, this variance deflation makes dynamic threshold bounds
  artificially narrow, triggering cascades of false positive alerts.
  Gate 1 mathematically audits estimator unbiasedness before telemetry
  ingestion.
* **First-Principles Mathematical Mechanism**:
  Why is the naive sample variance systematically biased downward?
  Because deviations are measured from the sample mean $\bar{x}$, which is
  itself estimated from the data, rather than the true population mean $\mu$.
  $$\sum_{i=1}^n (x_i - \bar{x})^2 = \sum_{i=1}^n (x_i - \mu)^2 - n(\bar{x} - \mu)^2$$
  Taking expectations across repeated samples:
  $$\mathbb{E}\left[\sum_{i=1}^n (x_i - \bar{x})^2\right] = \sum_{i=1}^n \mathbb{E}[(x_i - \mu)^2] - n\mathbb{E}[(\bar{x} - \mu)^2] = n\sigma^2 - n\left(\frac{\sigma^2}{n}\right) = (n - 1)\sigma^2$$
  Dividing by $n$ produces a biased expectation:
  $$\mathbb{E}[S_{\text{biased}}^2] = \mathbb{E}\left[\frac{1}{n}\sum_{i=1}^n (x_i - \bar{x})^2\right] = \frac{n-1}{n}\sigma^2$$
  To obtain an unbiased estimator $\mathbb{E}[S^2] = \sigma^2$, we must
  divide by $n - 1$:
  $$S^2 = \frac{1}{n - 1}\sum_{i=1}^n (x_i - \bar{x})^2 = \frac{n}{n - 1} S_{\text{biased}}^2$$
  For our sample ($n=7$, $\sum (x_i - \bar{x})^2 = 2084.8571$):
  $$s_{\text{biased}}^2 = \frac{2084.8571}{7} \approx 297.8367\text{ ms}^2, \qquad s^2 = \frac{2084.8571}{6} \approx 347.4762\text{ ms}^2$$
  $$s^2 / s_{\text{biased}}^2 = 7/6 \approx 1.166667$$
* **The Automated Assertion Contract (`[PASS]` / `[FAIL]` Criteria)**:
  * Degrees of Freedom Invariant: $df == n - 1 == 6$ strictly (`[PASS]`).
  * Bessel Scale Factor Invariant: $|s_{\text{unbiased}}^2 / s_{\text{biased}}^2 - 7/6| < 10^{-12}$ (`[PASS]`).
  * Exact Unbiased Variance: $|s^2 - 347.476190| < 10^{-5}\text{ ms}^2$ (`[PASS]`).
  * Asymptotic Expectation Invariant: Across $10{,}000$ simulated draws from
    $\text{Normal}(58.857, 347.476)$, the empirical mean of $S^2$ satisfies
    $|\bar{s}^2 - 347.476| \le 3.5\text{ ms}^2$ (`[PASS]`).
* **The Audit Prompt**:
  ```text
  AUDIT GATE 1 (Bessel's Correction & Estimator Unbiasedness):
  Data: [42, 58, 65, 48, 52, 98, 49], n = 7.
  1. Compute sum of squared deviations SS = sum((x_i - x_bar)^2).
  2. Compute biased variance s_biased^2 = SS / 7.
  3. Compute unbiased variance s^2 = SS / 6.
  4. Assert df == 6 and s^2 / s_biased^2 == 7/6.
  5. Assert s^2 == 347.4762 ms^2 and s == 18.6407 ms.
  Report [PASS] or [FAIL] with exact algebraic ratios.
  ```

---

### Gate 2: Student-$t$ vs. Asymptotic $Z$ Critical Value Audit

* **The Production Risk (Why This Gate Exists)**:
  LLMs are heavily pre-trained on generic text where "95% confidence" is
  synonymous with $\pm 1.96$. But for small sample sizes ($n=7$), the
  sampling distribution of the mean has heavy tails because $\sigma$ is
  unknown and must be estimated via $s$. Applying $Z = 1.960$ underestimates
  the margin of error by $24.8\%$ ($ME_Z = 13.81\text{ ms}$ vs.
  $ME_t = 17.24\text{ ms}$). An automated deployment pipeline using $Z$ will
  produce overly narrow confidence bounds, certifying unstable systems that
  subsequently fail in production. Gate 2 enforces the exact Student-$t$
  distribution.
* **First-Principles Mathematical Mechanism**:
  Let $Z = \frac{\bar{x} - \mu}{\sigma / \sqrt{n}} \sim \text{Normal}(0, 1)$.
  Let $V = \frac{(n-1)S^2}{\sigma^2} \sim \chi^2(n-1)$.
  By Cochran's theorem, sample mean $\bar{x}$ and sample variance $S^2$ are
  mutually independent for normally distributed data.
  The Student-$t$ test statistic is defined as the ratio:
  $$T = \frac{Z}{\sqrt{V / (n-1)}} = \frac{\frac{\bar{x} - \mu}{\sigma / \sqrt{n}}}{\sqrt{\frac{(n-1)S^2}{\sigma^2 (n-1)}}} = \frac{\bar{x} - \mu}{S / \sqrt{n}} \sim \text{Student-}t(df = n-1)$$
  For $df = 6$:
  * The density is $p(t) = \frac{\Gamma(7/2)}{\sqrt{6\pi}\,\Gamma(3)} \left(1 +
  \frac{t^2}{6}\right)^{-7/2}$.
  * The variance of the Student-$t$ distribution is $\frac{df}{df - 2} =
  \frac{6}{4} = 1.50$ (50% more variance than $\text{Normal}(0, 1)$!).
  * Critical value lookup: $t_{0.025, 6} = 2.44691$.
  * Margin of error comparison:
    $$ME_t = 2.44691 \times 7.0455 = 17.2398\text{ ms}$$
    $$ME_Z = 1.95996 \times 7.0455 = 13.8089\text{ ms}$$
    $$ME_t / ME_Z = 2.44691 / 1.95996 \approx 1.24845 \quad (24.85\% \text{ wider})$$
* **The Automated Assertion Contract (`[PASS]` / `[FAIL]` Criteria)**:
  * Critical Value Invariant: $|t^* - 2.44691| < 10^{-4}$ (`[PASS]`; any
    output using $1.960$ triggers immediate `[FAIL]`).
  * Exact Standard Error: $|SE - 7.045533| < 10^{-5}\text{ ms}$ (`[PASS]`).
  * Margin of Error Bound: $|ME - 17.2398| \le 0.0050\text{ ms}$ (`[PASS]`).
  * Small-Sample Tail Sizing: $ME / (1.960 \times SE) \ge 1.245$ (`[PASS]`).
* **The Audit Prompt**:
  ```text
  AUDIT GATE 2 (Student-t Distributional Tail Sizing):
  Sample: n = 7, s = 18.6407 ms.
  1. Compute SE = s / sqrt(7) ≈ 7.0455 ms.
  2. For 95% two-tailed confidence with df = 6, query exact quantile t_{0.025,
  6}.
  3. Verify t* == 2.44691 (reject 1.960).
  4. Compute margin of error ME = 2.44691 * 7.0455 = 17.2398 ms.
  5. Compute 95% CI: [58.8571 - 17.2398, 58.8571 + 17.2398] = [41.62, 76.10] ms.
  Report [PASS] or [FAIL] with critical value justification.
  ```

---

### Gate 3: Frequentist Coverage Simulation (10,000 Parallel Runs)

* **The Production Risk (Why This Gate Exists)**:
  Engineers frequently mistake confidence intervals for Bayesian posterior
  bounds, claiming that the true parameter has a 95% probability of residing
  within $[41.62, 76.10]$. In frequentist statistics, the true parameter is a
  fixed constant. The 95% label is an empirical performance guarantee about the
  *algorithm* across repeated sampling. If an agent does not understand this
  coverage contract, it cannot design valid continuous testing suites or A/B
  experiment stop rules. Gate 3 executes a 10,000-trial Monte Carlo audit to
  prove that the algorithm achieves nominal 95% coverage.
* **First-Principles Mathematical Mechanism**:
  We test the procedure by repeatedly generating synthetic datasets from a known
  population $\text{Normal}(\mu^* = 58.8571, \sigma^* = 18.6407)$ across
  $M = 10{,}000$ parallel simulations:
  For simulation $m = 1, \dots, 10{,}000$:
  1. Sample $n = 7$ values: $X_m = [x_1, \dots, x_7] \sim \text{Normal}(\mu^*,
  (\sigma^*)^2)$.
  2. Compute sample mean $\bar{x}_m$ and sample variance $s_m^2 =
  \frac{1}{6}\sum (x_i - \bar{x}_m)^2$.
  3. Compute $SE_m = s_m / \sqrt{7}$.
  4. Compute Student-$t$ interval:
     $$CI_{t, m} = [\bar{x}_m - 2.4469 SE_m, \; \bar{x}_m + 2.4469 SE_m]$$
  5. Compute naive $Z$ interval:
     $$CI_{Z, m} = [\bar{x}_m - 1.9600 SE_m, \; \bar{x}_m + 1.9600 SE_m]$$
  6. Check hits: $h_{t, m} = \mathbb{I}(\mu^* \in CI_{t, m})$ and $h_{Z, m} =
  \mathbb{I}(\mu^* \in CI_{Z, m})$.
* **The Automated Assertion Contract (`[PASS]` / `[FAIL]` Criteria)**:
  * Student-$t$ Empirical Coverage: Across $M=10{,}000$ runs, coverage
    $\hat{C}_t = \frac{1}{M}\sum h_{t, m}$ must satisfy $|\hat{C}_t - 0.9500| \le 0.0050$
    ($[94.50\%, 95.50\%]$) (`[PASS]`).
  * Naive $Z$ Undercoverage Detection: Naive coverage satisfies $\hat{C}_Z \le
  0.9000$
    (empirically $\approx 89.2\%$, proving $Z$ undercovers) (`[PASS]`).
  * Unbiased Mean Estimation: Mean estimation error across trials
    $|\frac{1}{M}\sum (\bar{x}_m - \mu^*)| \le 0.050\text{ ms}$ (`[PASS]`).
* **The Audit Prompt**:
  ```text
  AUDIT GATE 3 (Frequentist Coverage Across 10,000 Universes):
  Target: Evaluate nominal 95% coverage of Student-t vs. Z on n = 7.
  Ground truth: mu* = 58.8571, sigma* = 18.6407.
  For m = 1 to 10,000:
    1. Draw 7 samples from Normal(mu*, sigma*^2).
    2. Compute x_bar, s (df=6), and SE = s / sqrt(7).
    3. Check if mu* is within x_bar ± 2.4469 * SE (Student-t).
    4. Check if mu* is within x_bar ± 1.9600 * SE (Z-score).
  Assert Student-t coverage is 95.0% ± 0.5% and Z coverage is < 90.0%.
  Report [PASS] or [FAIL] with empirical coverage percentages.
  ```

---

### Gate 4: Asymmetric SLA Certification & Risk Decision Audit

* **The Production Risk (Why This Gate Exists)**:
  The most ubiquitous operational blunder in engineering is confusing a point
  estimate with compliance. Here, the sample mean is $\bar{x} = 58.86\text{
  ms}$,
  which is below the $60.0\text{ ms}$ SLA limit. An uncalibrated agent will
  declare: *"SLA passed! 58.86 ms < 60 ms."* But in production software,
  shipping an API that breaches SLA incurs severe contractual penalties,
  customer
  churn, and pager escalations ($C_{\text{breach}} \gg C_{\text{hold}}$).
  Because the sample size is small ($n=7$), the upper confidence bound extends
  to
  $76.10\text{ ms}$. Gate 4 enforces formal hypothesis testing and one-sided
  tolerance limits before certifying production readiness.
* **First-Principles Mathematical Mechanism**:
  1. **Formal One-Tailed Hypothesis Test**:
     To certify that the service complies with the SLA ($\mu < 60.0	ext{ ms}$),
     the burden of proof must be on compliance:
     * Null Hypothesis $H_0: \mu \ge 60.0\text{ ms}$ (Service violates SLA).
     * Alternative Hypothesis $H_1: \mu < 60.0\text{ ms}$ (Service satisfies SLA).
     Test statistic:
     $$t = \frac{\bar{x} - \mu_0}{SE} = \frac{58.8571 - 60.0000}{7.0455} = \frac{-1.1429}{7.0455} \approx -0.1622$$
     Degrees of freedom: $df = 6$.
     The $p$-value for a lower-tailed test is:
     $$p = P(T_6 \le -0.1622) = F_t(-0.1622; df=6) \approx 0.4384 \quad (43.84\%)$$
     At standard significance level $\alpha = 0.05$, we can only reject $H_0$
     if $p < 0.05$. Because $p = 0.4384 \gg 0.05$, there is zero statistically
     compelling evidence that the true latency is below 60 ms.
  2. **One-Sided 95% Upper Confidence Bound**:
     For a one-sided $95\%$ upper bound ($t_{0.05, 6} = 1.94318$):
     $$\text{Upper Bound} = \bar{x} + t_{0.05, 6} \times SE = 58.8571 + 1.94318(7.0455) = 58.8571 + 13.6907 = 72.5478\text{ ms}$$
     Because $72.55\text{ ms} > 60.0\text{ ms}$, we cannot certify that the
     system meets the SLA.
* **The Automated Assertion Contract (`[PASS]` / `[FAIL]` Criteria)**:
  * Exact Test Statistic: $|t - (-0.1622)| \le 0.0020$ (`[PASS]`).
  * Exact $p$-Value Calculation: $|p - 0.4384| \le 0.0050$ (`[PASS]`).
  * Upper Bound Exceedance: Upper 95% bound $72.55\text{ ms} > 60.0\text{ ms}$
  verified (`[PASS]`).
  * Operational Release Invariant: `action == REJECT_SLA_CERTIFICATION` or
    `INSUFFICIENT_EVIDENCE` strictly (`[PASS]`; any agent output recommending
    `PASS` or `SHIP` triggers immediate `[FAIL]`).
* **The Audit Prompt**:
  ```text
  AUDIT GATE 4 (Frequentist SLA Certification Audit):
  Data: n = 7, x_bar = 58.8571 ms, SE = 7.0455 ms, SLA limit = 60.0 ms.
  1. Formulate H0: mu >= 60.0 vs H1: mu < 60.0.
  2. Compute test statistic t = (58.8571 - 60.0) / 7.0455 = -0.1622.
  3. Compute p-value for Student-t with df = 6: verify p ≈ 0.4384.
  4. Compute 95% one-sided upper confidence bound:
     x_bar + 1.9432 * SE = 72.55 ms.
  5. Evaluate decision rule: Is upper bound < 60.0 ms?
  Assert action == REJECT_SLA_CERTIFICATION. Report [PASS] or [FAIL].
  ```


## Part 5: Naive AI Prompt vs. Calibrated Epistemic Specification

| Architectural Dimension | Naive Conversational Prompt | Calibrated Epistemic Specification |
| :--- | :--- | :--- |
| **Variance Denominator** | Divides by $n=7$ (biased: underestimates variance by $16.7\%$) | Mandates $n-1=6$ (Bessel's unbiased correction) |
| **Standard Error** | Conflates $SE$ with $s$ ($SE = 17.26\text{ ms}$, error of $2.45\times$) | Correctly calculates $SE = s/\sqrt{n} = 7.05\text{ ms}$ |
| **Sampling Distribution** | Uses asymptotic $Z = 1.96$ ($89.2\%$ real coverage) | Enforces Student's $t$ with $df=6$ ($t^* = 2.447$, true $95\%$ coverage) |
| **Interval Width** | $[25.03, 92.69]\text{ ms}$ ($\text{width} = 67.66\text{ ms}$) | $[41.62, 76.10]\text{ ms}$ ($\text{width} = 34.48\text{ ms}$) |
| **Philosophical Meaning** | Hallucinates: "95% probability true mean is inside" | Calibrated: "Procedure covers $\mu$ in 95% of repeated runs" |
| **Decision Verification** | Falsely claims SLA passed because $\bar{x} < 60$ | Rejects SLA certification because upper bound ($76.10$) exceeds SLA |


## Part 6: 🎯 Check Your Understanding — 3-Tier "Audit the AI" Exercises

### 🟢 Tier 1: Spot the AI Hallucination
An engineer asks an AI model to evaluate 5 memory consumption benchmarks (in
MB):
`[102, 98, 105, 110, 95]`.
The AI replies:
> *"The average memory is $102.0\text{ MB}$ with standard deviation $5.74\text{
MB}$.
> Using the 95% confidence interval formula $\bar{x} \pm 1.96 \cdot \sigma$, the
> interval is $[90.75\text{ MB}, 113.25\text{ MB}]$. There is a 95% probability
> that the true mean memory consumption is between 90.75 MB and 113.25 MB."*

List three critical statistical errors in this statement.

<details>
<summary><b>Click here to view the Tier 1 Solution...</b></summary>

1. **Standard Error Omission**:
   The AI used the sample standard deviation $s = 5.74$ directly in the
   confidence
   interval formula, forgetting to divide by $\sqrt{n} = \sqrt{5} \approx
   2.236$.
   The actual standard error is $SE = 5.74 / 2.236 \approx 2.57\text{ MB}$.
2. **$Z$ vs. $t$ Critical Value**:
   For $n=5$, degrees of freedom $df = 4$. The critical value is
   $t_{0.025, 4} = 2.776$, not $1.96$. Using $1.96$ severely undercovers small
   samples.
3. **The Frequentist Definition Fallacy**:
   Claiming there is a "95% probability the true mean is between..." is the
   inverse probability fallacy. A frequentist CI describes the procedure's
   coverage across hypothetical repetitions, not the probability of the fixed
   parameter.
</details>

---

### 🟡 Tier 2: Refactor the Flawed Specification
An engineer prompts:
```text
We benchmarked 10 cold-start boot times for our serverless container:
[310, 420, 290, 380, 450, 330, 390, 410, 360, 400] ms.
Calculate the 99% confidence interval.
```
Refactor this into a calibrated specification, calculate the degrees of
freedom, standard error, and exact Student-$t$ interval.

<details>
<summary><b>Click here to view the Tier 2 Solution...</b></summary>

```yaml
frequentist_specification:
  data: [310, 420, 290, 380, 450, 330, 390, 410, 360, 400]
  sample_size: 10
  degrees_of_freedom: "n - 1 = 9"
  confidence_level: 0.99
  critical_value: "t_{0.005, 9} = 3.2498"
```
**Analytical Derivation**:
* Sample mean: $\bar{x} = 374.0\text{ ms}$
* Sum of squared deviations: $\sum (x_i - \bar{x})^2 = 23{,}040\text{ ms}^2$
* Unbiased sample variance: $s^2 = 23{,}040 / 9 \approx 2560.0\text{ ms}^2$
* Sample standard deviation: $s = \sqrt{2560} \approx 50.596\text{ ms}$
* Standard error: $SE = 50.596 / \sqrt{10} \approx 16.000\text{ ms}$
* Margin of error: $ME = 3.2498 \times 16.000 = 51.997\text{ ms}$
* 99% Confidence Interval: $[374.0 - 52.00, \; 374.0 + 52.00] = [322.00\text{
ms}, \; 426.00\text{ ms}]$
</details>

---

### 🔴 Tier 3: Architect an Autonomous Agent Contract
Design an automated frequentist CI/CD gate that audits candidate microservice
releases. The contract must:
1. Accept raw benchmark latency arrays ($n \ge 15$).
2. Reject any analysis that computes biased variance or uses $Z$-scores.
3. Certify the build ONLY if the one-sided 95% upper confidence bound on
   latency is strictly less than $100.0\text{ ms}$.

<details>
<summary><b>Click here to view the Tier 3 Solution...</b></summary>

```yaml
autonomous_release_gate_contract:
  contract_id: "frequentist_latency_certifier_v1"
  input_schema:
    latencies: "array[float]"
    min_sample_size: 15
    sla_max_mean_ms: 100.0
    alpha: 0.05
  verification_gates:
    gate_1_sample_size:
      check: "len(latencies) >= 15"
      on_fail: "ABORT_INSUFFICIENT_RUNS"
    gate_2_unbiased_variance:
      formula: "s2 = sum((x - mean)^2) / (n - 1)"
      ddof_requirement: 1
    gate_3_student_t_distribution:
      df_formula: "n - 1"
      critical_value: "scipy.stats.t.ppf(1 - alpha, df=n-1)"
    gate_4_upper_bound_decision:
      bound_formula: "mean + t_critical * (s / sqrt(n))"
      decision_rule: >
        IF upper_bound < 100.0:
          ACTION: "CERTIFY_RELEASE"
        ELSE:
          ACTION: "BLOCK_RELEASE_SLA_VIOLATION"
```
</details>


## Key Takeaways & Course Navigation

1. **Spread vs. Precision**: Sample standard deviation $s$ measures how data
   scatters. Standard error $SE = s/\sqrt{n}$ measures how our estimate of the
   mean scatters. Confusing the two is a classic AI hallucination.
2. **Student's $t$ is Mandatory for Small Samples**: When $\sigma$ is estimated
   via $s$, the sampling distribution has fat tails. Always use $t_{df=n-1}$,
   never asymptotic $Z = 1.96$.
3. **Frequentist CI Does Not Mean Probability of $\mu$**: A frequentist CI
   tells you how the procedure behaves across parallel universes, not the
   posterior probability of the parameter.

---

### Course Navigation
* **⬅️ [Previous: Ch 02: Continuous Normal Conjugacy &
Precision](02_normal_conjugate_updating.ipynb)**
* **➡️ [Next: Ch 04: Calculating a Mean in a Bayesian
World](04_calculating_a_mean_bayesian.ipynb)**
* **Companion Tracks**: [📖 Conceptual](../conceptual/03_calculating_a_mean_frequentist.ipynb) | [🐍 Python](../python/03_calculating_a_mean_frequentist.ipynb) | [🔵 R](../r/03_calculating_a_mean_frequentist.ipynb)
